### 策略名称: 15分钟截面价格排序因子 (15-Minute Cross-Sectional Price Rank Factor)

**策略概述:**
本策略基于高频快照数据，构建每 15 分钟频率的价格截面排序因子。
首先计算盘口一档买卖中间价，并提取每个 15 分钟时间窗口的期末（最后一笔）中间价。
随后在每个时间截面上，对所有标的按价格从高到低进行排序并归一化。
逻辑上，该因子赋予低价股更高的因子值（接近 1），高价股更低的因子值（接近 0），体现某种反转或低价股偏好特征。

---

**数学逻辑 (Mathematical Logic):**
1.  **中间价计算 (Mid Price):**
    $$ P_{mid} = \frac{Ask_{1} + Bid_{1}}{2} $$
2.  **窗口采样 (Window Sampling):**
    将交易时间划分为 15 分钟窗口（如 09:30-09:45 标记为 94500），取窗口内最后一个时间点的中间价作为 $P_{close}$。
3.  **截面排序 (Cross-sectional Rank):**
    在每个 `trading_day` 和 `time_segment` 分组内，按 $P_{close}$ **降序**排列 (Descending)。
    - $Rank_{desc} = 1$: 截面内价格最高。
    - $Rank_{desc} = N$: 截面内价格最低 ($N$ 为该截面标的总数)。
4.  **归一化公式 (Normalization):**
    $$ Factor = 1.0 - \frac{N - Rank_{desc}}{N - 1} $$
    - **推导:**
        - 若价格最高 ($Rank_{desc}=1$): $Factor = 1 - \frac{N-1}{N-1} = 0$
        - 若价格最低 ($Rank_{desc}=N$): $Factor = 1 - 0 = 1$

---

**Args:**
* `datasource` (str): 数据源表名 (e.g., `'cpt_dwc_2026_stock_hs300_snapshot'`)
* `start_date` (str): 开始日期 `'YYYY-MM-DD HH:MM:SS'`
* `end_date` (str): 结束日期 `'YYYY-MM-DD HH:MM:SS'`

**Returns:**
* `pd.DataFrame`: 因子数据，包含 columns `['date', 'instrument', 'factor']`
    * 其中 `date` 为每个 15 分钟窗口的结束时间点。
    * `factor` 为归一化后的数值 [0, 1]。
"""


In [1]:
def main(datasource, start_date, end_date):
    """
    factor function
    构建 15分钟频率的价格排序因子 (Price Rank Factor)
    因子定义：在每个 15 分钟截面内，按价格从高到低排序并归一化到 [0, 1]。
    - 价格越高，factor 越接近 1
    - 价格越低，factor 越接近 0

    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import pandas as pd
    import dai

    sql = f"""
    -- 优化设置
    SET preserve_insertion_order=false;
    SET threads=4;

    WITH cte_snapshot AS (
        SELECT
            date,
            instrument_id,
            (ask_price1 + bid_price1) / 2.0 AS mid_price,  -- 使用中间价减少噪音
            strftime(date, '%Y-%m-%d') AS trading_day,

            CASE
                -- 上午
                WHEN strftime(date, '%H%M') >= '0930' AND strftime(date, '%H%M') < '0945' THEN 94500
                WHEN strftime(date, '%H%M') >= '0945' AND strftime(date, '%H%M') < '1000' THEN 100000
                WHEN strftime(date, '%H%M') >= '1000' AND strftime(date, '%H%M') < '1015' THEN 101500
                WHEN strftime(date, '%H%M') >= '1015' AND strftime(date, '%H%M') < '1030' THEN 103000
                WHEN strftime(date, '%H%M') >= '1030' AND strftime(date, '%H%M') < '1045' THEN 104500
                WHEN strftime(date, '%H%M') >= '1045' AND strftime(date, '%H%M') < '1100' THEN 110000
                WHEN strftime(date, '%H%M') >= '1100' AND strftime(date, '%H%M') < '1115' THEN 111500
                WHEN strftime(date, '%H%M') >= '1115' AND strftime(date, '%H%M') <= '1130' THEN 113000

                -- 下午
                WHEN strftime(date, '%H%M') >= '1300' AND strftime(date, '%H%M') < '1315' THEN 131500
                WHEN strftime(date, '%H%M') >= '1315' AND strftime(date, '%H%M') < '1330' THEN 133000
                WHEN strftime(date, '%H%M') >= '1330' AND strftime(date, '%H%M') < '1345' THEN 134500
                WHEN strftime(date, '%H%M') >= '1345' AND strftime(date, '%H%M') < '1400' THEN 140000
                WHEN strftime(date, '%H%M') >= '1400' AND strftime(date, '%H%M') < '1415' THEN 141500
                WHEN strftime(date, '%H%M') >= '1415' AND strftime(date, '%H%M') < '1430' THEN 143000
                WHEN strftime(date, '%H%M') >= '1430' AND strftime(date, '%H%M') < '1445' THEN 144500
                WHEN strftime(date, '%H%M') >= '1445' AND strftime(date, '%H%M') < '1457' THEN 150000
                ELSE -1
            END AS time_segment
        FROM {datasource}
    ),

    cte_filtered AS (
        SELECT *
        FROM cte_snapshot
        WHERE time_segment != -1
          AND mid_price IS NOT NULL
          AND mid_price > 0
    ),

    -- 每个15分钟窗口取收盘中间价（窗口最后一笔）
    cte_window AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,
            argMax(mid_price, date) AS close_mid
        FROM cte_filtered
        GROUP BY instrument_id, trading_day, time_segment
    ),

    -- 截面内按价格从高到低排序
    cte_rank AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,
            close_mid,
            row_number() OVER (
                PARTITION BY trading_day, time_segment
                ORDER BY close_mid DESC, instrument_id
            ) AS rn_desc,
            count(*) OVER (
                PARTITION BY trading_day, time_segment
            ) AS cnt_cs
        FROM cte_window
    ),

    cte_factor AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,
            -- 归一化到 [0,1]，最高价=1，最低价=0
            CASE
                WHEN cnt_cs <= 1 THEN 0.5
                ELSE 1.0 - (cnt_cs - rn_desc) * 1.0 / (cnt_cs - 1)
            END AS factor
        FROM cte_rank
    )

    SELECT
        CAST(CONCAT(
            f.trading_day, ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.factor
    FROM cte_factor f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={"date": [start_date, end_date]}).df()
    return df


if __name__ == "__main__":
    """
    开发调试专用模块：分块循环回测引擎
    """
    from bigmodule import M
    import pandas as pd
    import structlog
    import gc

    logger = structlog.get_logger()
    datasource = "cpt_dwc_2026_stock_hs300_snapshot"

    full_start_date = "2023-01-01"
    full_end_date = "2023-06-01"

    date_ranges = pd.date_range(start=full_start_date, end=full_end_date, freq="MS")
    all_results = []

    logger.info(f"🚀 Starting Price Rank Factor Backtest: {full_start_date} to {full_end_date}")

    for start_dt in date_ranges:
        current_start = start_dt.strftime("%Y-%m-%d 00:00:00")
        current_end = (start_dt + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d 23:59:59")
        logger.info(f"Processing Chunk: {current_start} => {current_end}")

        try:
            df_chunk = main(datasource, current_start, current_end)

            if df_chunk is not None and not df_chunk.empty:
                all_results.append(df_chunk)
                logger.info(f"✅ Chunk Done. Rows: {len(df_chunk)}")
            else:
                logger.warning(f"⚠️ Chunk Empty: {current_start}")

            del df_chunk
            gc.collect()

        except Exception as e:
            logger.error(f"❌ Error in chunk {current_start}: {e}")

    if all_results:
        logger.info("🧩 Concatenating all chunks...")
        final_data = pd.concat(all_results, ignore_index=True)
        final_data.sort_values(by=["date", "instrument"], inplace=True)

        logger.info(f"🎉 All Done! Final Shape: {final_data.shape}")
        logger.info(f"Sample:\n{final_data.head()}")

        logger.info("📊 Starting Evaluation...")
        try:
            _ = M.eval_dwc._latest(data=final_data)
        except Exception as e:
            logger.warning(f"Evaluation failed (local env might miss modules): {e}")
            print("Data preview:", final_data.head())
    else:
        logger.error("No data generated.")


[2026-02-06 11:02:01] [info     ] 🚀 Starting Price Rank Factor Backtest: 2023-01-01 to 2023-06-01
[2026-02-06 11:02:01] [info     ] Processing Chunk: 2023-01-01 00:00:00 => 2023-01-31 23:59:59
[2026-02-06 11:02:06] [info     ] ✅ Chunk Done. Rows: 76641
[2026-02-06 11:02:06] [info     ] Processing Chunk: 2023-02-01 00:00:00 => 2023-02-28 23:59:59
[2026-02-06 11:02:12] [info     ] ✅ Chunk Done. Rows: 95905
[2026-02-06 11:02:12] [info     ] Processing Chunk: 2023-03-01 00:00:00 => 2023-03-31 23:59:59
[2026-02-06 11:02:19] [info     ] ✅ Chunk Done. Rows: 110198
[2026-02-06 11:02:19] [info     ] Processing Chunk: 2023-04-01 00:00:00 => 2023-04-30 23:59:59
